In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import os
import sys
import glob
sys.path.append('modules')
import globalvars

path_to_data = globalvars.path_to_data

In [2]:
# -----------------------------
# Settings
# -----------------------------
START_DATE = "2000-01-01"
END_DATE   = "2019-12-31"
DAYS_PER_JOB = 10

varname = "uv1000" ## 'ivt', 'freezing_level', 'uv'

# -----------------------------
# Get SLURM task id
# -----------------------------
task_id = 0

# -----------------------------
# Build date list
# -----------------------------
dates = pd.date_range(START_DATE, END_DATE, freq="D")

start_idx = task_id * DAYS_PER_JOB
end_idx   = start_idx + DAYS_PER_JOB

if start_idx >= len(dates):
    print("No dates assigned to this task.")
    sys.exit()

date_subset = dates[start_idx:end_idx]

print(f"Processing dates: {date_subset}")

Processing dates: DatetimeIndex(['2000-01-01', '2000-01-02', '2000-01-03', '2000-01-04',
               '2000-01-05', '2000-01-06', '2000-01-07', '2000-01-08',
               '2000-01-09', '2000-01-10'],
              dtype='datetime64[ns]', freq='D')


In [3]:
# -----------------------------
# Loop over dates
# -----------------------------
for dt in date_subset[0:1]:

    date = dt.strftime("%Y%m%d")

    out_name = os.path.join(
        path_to_data,
        f"preprocessed/GEFSv12_reforecast/{varname}_final/"
        f"GEFSv12_reforecast_{varname}_{date}.nc"
    )

    # # -----------------------------
    # # Restart-safe check
    # # -----------------------------
    # if os.path.exists(out_name):
    #     print(f"Skipping {date} (already processed)")
    #     continue

    # -----------------------------
    # Load all forecast step files
    # -----------------------------
    fname_pattern = os.path.join(
        path_to_data,
        f'preprocessed/GEFSv12_reforecast/{varname}/{date}_{varname}_F*.nc'
    )
    files = glob.glob(fname_pattern)

    if len(files) == 0:
        print(f"No files found for {date}, skipping")
        continue
        
    forecast = xr.open_mfdataset(
        fname_pattern,
        engine="netcdf4",
        concat_dim="step",
        combine="nested",
        decode_timedelta=True,
        coords="minimal",
        data_vars="minimal",
        compat="override",
        parallel=False
    )
    
    # Make sure forecast steps are sorted
    forecast = forecast.sortby("step")

    # calculate wind magnitude
    if varname == 'uv1000':
        forecast["uv"] = np.sqrt(forecast['u']**2 + forecast['v']**2)

    # -----------------------------
    # Update variable metadata
    # -----------------------------
    var_attrs = {
        "step": {"long_name": "time since init_time"},
        "ivt":  {"long_name": "integrated water vapor transport",
                 "units": "kg m-1 s-1"},
        "ivtu": {"long_name": "zonal integrated water vapor transport",
                 "units": "kg m-1 s-1"},
        "ivtv": {"long_name": "meridional integrated water vapor transport",
                 "units": "kg m-1 s-1"},
        "uv": {"long_name": "wind magnitude",
               "units": "m s-1"},
    }
    
    for var, attrs in var_attrs.items():
        if var in forecast:
            forecast[var].attrs.update(attrs)
    
    # Add some global attributes (optional)
    forecast.attrs.update({
        "description": f"GEFSv12 reforecast {varname} fields merged into a single file",
        "init_date": date,
    })
    
    
    # -----------------------------
    # Rename dimension (if present)
    # -----------------------------
    if "time" in forecast.coords:
        forecast = forecast.rename({"time": "init_time"})
    else:
        np_date = pd.to_datetime(forecast.attrs['init_date'], format='%Y%m%d').to_datetime64()
        forecast = forecast.assign_coords(init_time=np_date)
    
    
    # -----------------------------
    # Drop unneeded variables
    # -----------------------------
    forecast = forecast.drop_vars("surface", errors="ignore")

forecast

<xarray.Dataset> Size: 646MB
Dimensions:        (number: 5, step: 80, latitude: 281, longitude: 479)
Coordinates:
  * number         (number) int64 40B 0 1 2 3 4
  * step           (step) timedelta64[ns] 640B 0 days 03:00:00 ... 10 days 00...
  * latitude       (latitude) float64 2kB 70.0 69.75 69.5 69.25 ... 0.5 0.25 0.0
  * longitude      (longitude) float64 4kB -179.5 -179.2 -179.0 ... -60.25 -60.0
    init_time      datetime64[ns] 8B ...
    isobaricInhPa  float64 8B ...
    valid_time     (step) datetime64[ns] 640B dask.array<chunksize=(8,), meta=np.ndarray>
Data variables:
    u              (number, step, latitude, longitude) float32 215MB dask.array<chunksize=(5, 8, 281, 479), meta=np.ndarray>
    v              (number, step, latitude, longitude) float32 215MB dask.array<chunksize=(5, 8, 281, 479), meta=np.ndarray>
    uv             (number, step, latitude, longitude) float32 215MB dask.array<chunksize=(5, 8, 281, 479), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          2
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    history:                 2024-09-09T10:26 GRIB to CDM+CF via cfgrib-0.9.1...
    description:             GEFSv12 reforecast uv1000 fields merged into a s...
    init_date:               20000101